# Official final ensemble reproduction

This notebook reproduces the three official final submissions used in our DLMMDD Synthetic Source Attribution Challenge solution.

The notebook trains or regenerates the branch-level prediction files:

1. `arcface_base`
2. `ssl_pre2_base`
3. `convnextv2_base_512_arcface_ce60arc40`
4. `512_arcface_base_postproc_aug_v1_inferacc`
5. `arcface_base_old_ft768_v1`

The final ensemble cell generates:

- `official_final_A_submission.csv`
- `official_final_B_submission.csv`
- `official_final_C_submission.csv`

Expected input files are placed under `./Data/`, and generated outputs are saved under `./ensemble_artifacts/`.

### 1. ArcFace_base

In [ ]:
%%time
# =========================================================
# Unified training script (ensemble-ready)
#   - saves fold-wise probs
#   - saves final oof/test probs
#   - avoids overwriting by OUTPUT_PREFIX
#   - supports:
#       "arcface" / "supcon" / "ssl" / "vit_ce"
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import copy
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# =========================================================
# Select model kind
# =========================================================
MODEL_KIND = "arcface"   # "arcface" / "supcon" / "ssl" / "vit_ce"

# IMPORTANT:
# Give every experiment a unique output prefix
OUTPUT_PREFIX = "arcface_base"

# =========================================================
# Config
# =========================================================
@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"

    save_dir: str = "./ensemble_artifacts"

    # convnext family
    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"

    # vit family
    vit_model_name: str = "vit_base_patch16_384"

    image_size: int = 384
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20
    ft_epochs: int = 20
    pre_epochs: int = 4

    train_bs: int = 16
    valid_bs: int = 32
    ssl_bs: int = 8
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    ssl_lr_backbone: float = 1e-4
    ssl_lr_proj: float = 3e-4

    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    # ArcFace
    arc_s: float = 30.0
    arc_m: float = 0.30
    ce_weight_arc: float = 0.6
    arc_weight: float = 0.4

    # SupCon
    temperature: float = 0.07
    ce_weight_sup: float = 0.7
    supcon_weight: float = 0.3

    # Early stopping
    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0

    tta_count: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)

# =========================================================
# Output helpers
# =========================================================
def out_path(name):
    return os.path.join(cfg.save_dir, f"{OUTPUT_PREFIX}_{name}")

def best_model_path(fold):
    return out_path(f"best_fold{fold}.pth")

def fold_val_probs_path(fold):
    return out_path(f"fold{fold}_val_probs.npy")

def fold_test_probs_path(fold):
    return out_path(f"fold{fold}_test_probs.npy")

def final_oof_probs_path():
    return out_path("oof_probs.npy")

def final_test_probs_path():
    return out_path("test_probs.npy")

def final_submission_path():
    return out_path("submission.csv")

def metadata_path():
    return out_path("metadata.json")

def fold_scores_path():
    return out_path("fold_scores.csv")

# =========================================================
# Utils
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

def make_abs_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)

# =========================================================
# Read data
# =========================================================
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train.csv must contain 'y' or 'TARGET'.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("MODEL_KIND:", MODEL_KIND)
print("OUTPUT_PREFIX:", OUTPUT_PREFIX)
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

# =========================================================
# Transforms
# =========================================================
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def get_ssl_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.60, 1.0), ratio=(0.85, 1.15), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=12, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.25),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.35),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_train_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.75, 1.0), ratio=(0.90, 1.10), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.ToGray(p=0.07),
        A.OneOf([
            A.Downscale(scale_range=(0.5, 0.85), p=1.0),
            A.Resize(int(img_size * 0.85), int(img_size * 0.85), p=1.0),
        ], p=0.12),
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size=384, tta_id=0):
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
            A.CenterCrop(height=img_size, width=img_size, p=1.0),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

# =========================================================
# Datasets
# =========================================================
class BaseImageDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])
        return image, int(row["label"])

class SupConDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False, two_views=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.two_views = two_views

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.is_test:
            image = self.transform(image=image)["image"]
            return image, int(row["ID"])

        label = int(row["label"])
        if self.two_views:
            image1 = self.transform(image=image)["image"]
            image2 = self.transform(image=image)["image"]
            return image1, image2, label
        else:
            image = self.transform(image=image)["image"]
            return image, label

class SSLDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        v1 = self.transform(image=image)["image"]
        v2 = self.transform(image=image)["image"]
        return v1, v2

# =========================================================
# Common blocks
# =========================================================
class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = score > self.best_score + self.min_delta if self.mode == "max" else score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

# =========================================================
# ArcFace model
# =========================================================
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0, 1.0)
        if labels is None:
            return cosine * self.s
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s

class ArcFaceModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, arc_s=30.0, arc_m=0.30, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(embedding_dim, num_classes, s=arc_s, m=arc_m)

    def forward(self, x, labels=None):
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb = self.dropout(emb)
        ce_logits = self.ce_head(emb)
        if labels is not None:
            arc_logits = self.arc_head(emb, labels)
            return ce_logits, arc_logits, emb
        arc_logits = self.arc_head(emb, None)
        return ce_logits, arc_logits, emb

# =========================================================
# SupCon model
# =========================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        bsz = features.shape[0]
        n_views = features.shape[1]

        features = F.normalize(features, dim=2)
        features = torch.cat(torch.unbind(features, dim=1), dim=0)

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        mask = mask.repeat(n_views, n_views)

        logits = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        logits_mask = torch.ones_like(mask)
        logits_mask.scatter_(1, torch.arange(bsz * n_views, device=device).view(-1, 1), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        mask_sum = torch.clamp(mask.sum(dim=1), min=1.0)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask_sum
        loss = -mean_log_prob_pos
        loss = loss.view(n_views, bsz).mean()
        return loss

class SupConModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)

    def forward_features(self, x):
        feat = self.backbone(x)
        emb = self.neck(feat)
        return emb

    def forward(self, x):
        emb = self.forward_features(x)
        logits = self.ce_head(self.dropout(emb))
        return logits, emb

# =========================================================
# SSL models
# =========================================================
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)

class PredictionHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=256, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class SSLPretrainModel(nn.Module):
    def __init__(self, model_name, embedding_dim=512):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.encoder = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.GELU(),
        )
        self.projector = ProjectionHead(embedding_dim, hidden_dim=1024, out_dim=embedding_dim)
        self.predictor = PredictionHead(embedding_dim, hidden_dim=256, out_dim=embedding_dim)

    def forward_backbone(self, x):
        feat = self.backbone(x)
        emb = self.encoder(feat)
        return emb

    def forward_ssl(self, x1, x2):
        f1 = self.forward_backbone(x1)
        f2 = self.forward_backbone(x2)
        z1 = self.projector(f1)
        z2 = self.projector(f2)
        p1 = self.predictor(z1)
        p2 = self.predictor(z2)
        return p1, p2, z1.detach(), z2.detach()

class SSLFineTuneModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.neck(feat)
        logits = self.head(self.dropout(emb))
        return logits, emb

# =========================================================
# ViT CE model
# =========================================================
class ViTCEModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )
        backbone_out = self.backbone.num_features
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(backbone_out, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.head(self.dropout(feat))
        return logits, feat

# =========================================================
# Common predict helpers
# =========================================================
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
supcon_criterion = SupConLoss(temperature=cfg.temperature)

@torch.no_grad()
def predict_probs_standard(model, loader, device):
    model.eval()
    all_probs, all_ids = [], []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            if MODEL_KIND == "arcface":
                ce_logits, arc_logits, _ = model(images, labels=None)
                logits = 0.5 * ce_logits + 0.5 * arc_logits
            else:
                logits, _ = model(images)

            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(ids.tolist() if torch.is_tensor(ids) else list(ids))

    return np.array(all_ids), np.concatenate(all_probs, axis=0)

@torch.no_grad()
def predict_with_tta_standard(model, df, device, is_test=True, tta_count=4):
    probs_list, labels_ref, ids_ref = [], None, None

    for tta_id in range(tta_count):
        ds = BaseImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False
        )

        if is_test:
            ids, probs = predict_probs_standard(model, dl, device)
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs, fold_labels = [], []
            for images, labels in dl:
                images = images.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    if MODEL_KIND == "arcface":
                        ce_logits, arc_logits, _ = model(images, labels=None)
                        logits = 0.5 * ce_logits + 0.5 * arc_logits
                    else:
                        logits, _ = model(images)
                    probs = torch.softmax(logits, dim=1)

                fold_probs.append(probs.cpu().numpy())
                fold_labels.append(labels.numpy())

            labels_ref = np.concatenate(fold_labels, axis=0)
            probs_list.append(np.concatenate(fold_probs, axis=0))

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)
    return (ids_ref, mean_probs) if is_test else (mean_probs, labels_ref)

# =========================================================
# ArcFace train/valid
# =========================================================
def train_arcface_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_ce = total_arc = total_correct = total_count = 0

    arc_criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (fused_logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_ce / total_count, total_arc / total_count, total_correct / total_count

@torch.no_grad()
def valid_arcface_one_epoch(model, loader, device):
    model.eval()
    arc_criterion = nn.CrossEntropyLoss()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(fused_logits, dim=1)
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# SupCon train/valid
# =========================================================
def train_supcon_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_ce = total_sup = total_correct = total_count = 0

    for image1, image2, labels in loader:
        image1 = image1.to(device, non_blocking=True)
        image2 = image2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits1, emb1 = model(image1)
            _, emb2 = model(image2)
            ce_loss = ce_criterion(logits1, labels)
            sup_loss = supcon_criterion(torch.stack([emb1, emb2], dim=1), labels)
            loss = cfg.ce_weight_sup * ce_loss + cfg.supcon_weight * sup_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_sup += sup_loss.item() * labels.size(0)
        total_correct += (logits1.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_ce / total_count, total_sup / total_count, total_correct / total_count

@torch.no_grad()
def valid_supcon_one_epoch(model, loader, device):
    model.eval()
    total_loss = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# ViT CE train/valid
# =========================================================
def train_vit_ce_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            loss = ce_criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def valid_vit_ce_one_epoch(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_labels, axis=0),
    )

# =========================================================
# SSL pretrain + fine-tune
# =========================================================
def negative_cosine_similarity(p, z):
    p = F.normalize(p, dim=1)
    z = F.normalize(z, dim=1)
    return -(p * z).sum(dim=1).mean()

def ssl_pretrain(model, loader, epochs, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    backbone_params = list(model.backbone.parameters()) + list(model.encoder.parameters())
    proj_params = list(model.projector.parameters()) + list(model.predictor.parameters())

    optimizer = torch.optim.AdamW(
        [{"params": backbone_params, "lr": cfg.ssl_lr_backbone},
         {"params": proj_params, "lr": cfg.ssl_lr_proj}],
        weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(loader), eta_min=cfg.min_lr
    )

    for epoch in range(epochs):
        running_loss, total = 0.0, 0
        t0 = time.time()

        for x1, x2 in loader:
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                p1, p2, z1, z2 = model.forward_ssl(x1, x2)
                loss = 0.5 * negative_cosine_similarity(p1, z2) + 0.5 * negative_cosine_similarity(p2, z1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * x1.size(0)
            total += x1.size(0)

        print(f"[SSL] Epoch {epoch+1:02d}/{epochs} | ssl_loss={running_loss/total:.4f} | time={time.time()-t0:.1f}s")

    return model

def train_ssl_finetune_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_correct = total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            loss = ce_criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def valid_ssl_finetune_one_epoch(model, loader, device):
    model.eval()
    total_loss = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# Optional SSL pretrain
# =========================================================
ssl_backbone_state = None
ssl_encoder_state = None

if MODEL_KIND == "ssl":
    print("\n" + "=" * 90)
    print("SSL PRETRAINING ON TRAIN ONLY")
    print("=" * 90)

    ssl_ds = SSLDataset(train_df, transform=get_ssl_transforms(cfg.image_size))
    ssl_loader = DataLoader(
        ssl_ds,
        batch_size=cfg.ssl_bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True
    )

    ssl_model = SSLPretrainModel(cfg.model_name, embedding_dim=cfg.embedding_dim).to(cfg.device)
    ssl_model = ssl_pretrain(ssl_model, ssl_loader, epochs=cfg.pre_epochs, device=cfg.device)

    ssl_backbone_state = copy.deepcopy(ssl_model.backbone.state_dict())
    ssl_encoder_state = copy.deepcopy(ssl_model.encoder.state_dict())

    del ssl_ds, ssl_loader, ssl_model
    gc.collect()
    torch.cuda.empty_cache()

# =========================================================
# CV
# =========================================================
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    print("\n" + "=" * 90)
    print(f"FOLD {fold+1}/{cfg.n_splits}")
    print("=" * 90)

    trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
    val_df = train_df.iloc[va_idx].reset_index(drop=True)

    if MODEL_KIND == "arcface":
        train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = ArcFaceModel(
            cfg.model_name, cfg.num_classes,
            embedding_dim=cfg.embedding_dim,
            arc_s=cfg.arc_s, arc_m=cfg.arc_m,
            dropout=cfg.dropout
        ).to(cfg.device)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.ce_head.parameters()) + list(model.arc_head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()
            tr_loss, tr_ce, tr_arc, tr_acc = train_arcface_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_ce, va_arc, va_acc, _, _ = valid_arcface_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, arc={tr_arc:.4f}) | "
                f"train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} (ce={va_ce:.4f}, arc={va_arc:.4f}) | "
                f"valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "supcon":
        train_ds = SupConDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False, two_views=True)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = SupConModel(cfg.model_name, cfg.num_classes, embedding_dim=cfg.embedding_dim, dropout=cfg.dropout).to(cfg.device)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.ce_head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()
            tr_loss, tr_ce, tr_sup, tr_acc = train_supcon_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_ce, va_acc, _, _ = valid_supcon_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, supcon={tr_sup:.4f}) | "
                f"train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "ssl":
        train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = SSLFineTuneModel(cfg.model_name, cfg.num_classes, embedding_dim=cfg.embedding_dim, dropout=cfg.dropout).to(cfg.device)

        model.backbone.load_state_dict(copy.deepcopy(ssl_backbone_state))
        try:
            model.neck[0].weight.data.copy_(ssl_encoder_state["0.weight"])
            model.neck[0].bias.data.copy_(ssl_encoder_state["0.bias"])
            model.neck[1].weight.data.copy_(ssl_encoder_state["1.weight"])
            model.neck[1].bias.data.copy_(ssl_encoder_state["1.bias"])
            model.neck[1].running_mean.data.copy_(ssl_encoder_state["1.running_mean"])
            model.neck[1].running_var.data.copy_(ssl_encoder_state["1.running_var"])
        except Exception as e:
            print("partial neck transfer skipped:", e)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.ft_epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.ft_epochs):
            t0 = time.time()
            tr_loss, tr_acc = train_ssl_finetune_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_acc, _, _ = valid_ssl_finetune_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.ft_epochs} | "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "vit_ce":
        train_ds = BaseImageDataset(
            trn_df,
            transform=get_train_transforms(cfg.image_size),
            is_test=False
        )
        valid_ds = BaseImageDataset(
            val_df,
            transform=get_valid_transforms(cfg.image_size),
            is_test=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=cfg.train_bs,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=True
        )
        valid_loader = DataLoader(
            valid_ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True
        )

        model = ViTCEModel(
            model_name=cfg.vit_model_name,
            num_classes=cfg.num_classes,
            dropout=cfg.dropout
        ).to(cfg.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr_backbone,
            weight_decay=cfg.weight_decay
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.epochs * len(train_loader),
            eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()

            tr_loss, tr_acc = train_vit_ce_one_epoch(
                model, train_loader, optimizer, scheduler, cfg.device
            )
            va_loss, va_acc, _, _ = valid_vit_ce_one_epoch(
                model, valid_loader, cfg.device
            )

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | "
                f"time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    else:
        raise ValueError("MODEL_KIND must be one of: arcface / supcon / ssl / vit_ce")

    model.load_state_dict(torch.load(best_path, map_location=cfg.device))

    val_tta_probs, val_tta_labels = predict_with_tta_standard(model, val_df, cfg.device, is_test=False, tta_count=cfg.tta_count)
    val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
    print(f"Fold {fold+1} TTA valid acc: {val_tta_acc:.5f}")
    oof_probs[va_idx] = val_tta_probs
    np.save(fold_val_probs_path(fold), val_tta_probs)

    _, fold_test_probs = predict_with_tta_standard(model, test_df, cfg.device, is_test=True, tta_count=cfg.tta_count)
    test_probs += fold_test_probs / cfg.n_splits
    np.save(fold_test_probs_path(fold), fold_test_probs)

    fold_scores.append({
        "fold": fold,
        "tta_valid_acc": float(val_tta_acc),
        "best_valid_acc": float(best_acc),
    })

    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    torch.cuda.empty_cache()

# =========================================================
# Save outputs
# =========================================================
oof_preds = oof_probs.argmax(1)
oof_acc = accuracy_score(train_df["label"].values, oof_preds)
print("\n" + "=" * 90)
print(f"OOF ACCURACY: {oof_acc:.6f}")
print("=" * 90)

np.save(final_oof_probs_path(), oof_probs)
np.save(final_test_probs_path(), test_probs)

submission = test_df[["ID"]].copy()
submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
submission.to_csv(final_submission_path(), index=False)

pd.DataFrame(fold_scores).to_csv(fold_scores_path(), index=False)

meta = {
    "model_kind": MODEL_KIND,
    "output_prefix": OUTPUT_PREFIX,
    "cfg": asdict(cfg),
    "oof_accuracy": float(oof_acc),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}
with open(metadata_path(), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"saved -> {final_oof_probs_path()}")
print(f"saved -> {final_test_probs_path()}")
print(f"saved -> {final_submission_path()}")
print(f"saved -> {fold_scores_path()}")
print(f"saved -> {metadata_path()}")
print(submission.head())

### 2. SSL_Pre2_base

In [ ]:
%%time
# =========================================================
# Unified training script (ensemble-ready)
#   - saves fold-wise probs
#   - saves final oof/test probs
#   - avoids overwriting by OUTPUT_PREFIX
#   - supports:
#       "arcface" / "supcon" / "ssl" / "vit_ce"
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import copy
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# =========================================================
# Select model kind
# =========================================================
MODEL_KIND = "ssl"   # "arcface" / "supcon" / "ssl" / "vit_ce"

# IMPORTANT:
# Give every experiment a unique output prefix
OUTPUT_PREFIX = "ssl_pre2_base"

# =========================================================
# Config
# =========================================================
@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"

    save_dir: str = "./ensemble_artifacts"

    # convnext family
    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"

    # vit family
    vit_model_name: str = "vit_base_patch16_384"

    image_size: int = 384
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20
    ft_epochs: int = 20
    pre_epochs: int = 2

    train_bs: int = 16
    valid_bs: int = 32
    ssl_bs: int = 8
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    ssl_lr_backbone: float = 1e-4
    ssl_lr_proj: float = 3e-4

    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    # ArcFace
    arc_s: float = 30.0
    arc_m: float = 0.30
    ce_weight_arc: float = 0.6
    arc_weight: float = 0.4

    # SupCon
    temperature: float = 0.07
    ce_weight_sup: float = 0.7
    supcon_weight: float = 0.3

    # Early stopping
    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0

    tta_count: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)

# =========================================================
# Output helpers
# =========================================================
def out_path(name):
    return os.path.join(cfg.save_dir, f"{OUTPUT_PREFIX}_{name}")

def best_model_path(fold):
    return out_path(f"best_fold{fold}.pth")

def fold_val_probs_path(fold):
    return out_path(f"fold{fold}_val_probs.npy")

def fold_test_probs_path(fold):
    return out_path(f"fold{fold}_test_probs.npy")

def final_oof_probs_path():
    return out_path("oof_probs.npy")

def final_test_probs_path():
    return out_path("test_probs.npy")

def final_submission_path():
    return out_path("submission.csv")

def metadata_path():
    return out_path("metadata.json")

def fold_scores_path():
    return out_path("fold_scores.csv")

# =========================================================
# Utils
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

def make_abs_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)

# =========================================================
# Read data
# =========================================================
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train.csv must contain 'y' or 'TARGET'.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("MODEL_KIND:", MODEL_KIND)
print("OUTPUT_PREFIX:", OUTPUT_PREFIX)
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

# =========================================================
# Transforms
# =========================================================
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def get_ssl_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.60, 1.0), ratio=(0.85, 1.15), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=12, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.25),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.35),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_train_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.75, 1.0), ratio=(0.90, 1.10), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.ToGray(p=0.07),
        A.OneOf([
            A.Downscale(scale_range=(0.5, 0.85), p=1.0),
            A.Resize(int(img_size * 0.85), int(img_size * 0.85), p=1.0),
        ], p=0.12),
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size=384, tta_id=0):
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
            A.CenterCrop(height=img_size, width=img_size, p=1.0),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

# =========================================================
# Datasets
# =========================================================
class BaseImageDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])
        return image, int(row["label"])

class SupConDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False, two_views=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.two_views = two_views

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.is_test:
            image = self.transform(image=image)["image"]
            return image, int(row["ID"])

        label = int(row["label"])
        if self.two_views:
            image1 = self.transform(image=image)["image"]
            image2 = self.transform(image=image)["image"]
            return image1, image2, label
        else:
            image = self.transform(image=image)["image"]
            return image, label

class SSLDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        v1 = self.transform(image=image)["image"]
        v2 = self.transform(image=image)["image"]
        return v1, v2

# =========================================================
# Common blocks
# =========================================================
class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = score > self.best_score + self.min_delta if self.mode == "max" else score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop

# =========================================================
# ArcFace model
# =========================================================
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0, 1.0)
        if labels is None:
            return cosine * self.s
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s

class ArcFaceModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, arc_s=30.0, arc_m=0.30, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(embedding_dim, num_classes, s=arc_s, m=arc_m)

    def forward(self, x, labels=None):
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb = self.dropout(emb)
        ce_logits = self.ce_head(emb)
        if labels is not None:
            arc_logits = self.arc_head(emb, labels)
            return ce_logits, arc_logits, emb
        arc_logits = self.arc_head(emb, None)
        return ce_logits, arc_logits, emb

# =========================================================
# SupCon model
# =========================================================
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        bsz = features.shape[0]
        n_views = features.shape[1]

        features = F.normalize(features, dim=2)
        features = torch.cat(torch.unbind(features, dim=1), dim=0)

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        mask = mask.repeat(n_views, n_views)

        logits = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()

        logits_mask = torch.ones_like(mask)
        logits_mask.scatter_(1, torch.arange(bsz * n_views, device=device).view(-1, 1), 0)
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        mask_sum = torch.clamp(mask.sum(dim=1), min=1.0)
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask_sum
        loss = -mean_log_prob_pos
        loss = loss.view(n_views, bsz).mean()
        return loss

class SupConModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)

    def forward_features(self, x):
        feat = self.backbone(x)
        emb = self.neck(feat)
        return emb

    def forward(self, x):
        emb = self.forward_features(x)
        logits = self.ce_head(self.dropout(emb))
        return logits, emb

# =========================================================
# SSL models
# =========================================================
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=1024, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)

class PredictionHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=256, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class SSLPretrainModel(nn.Module):
    def __init__(self, model_name, embedding_dim=512):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.encoder = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.GELU(),
        )
        self.projector = ProjectionHead(embedding_dim, hidden_dim=1024, out_dim=embedding_dim)
        self.predictor = PredictionHead(embedding_dim, hidden_dim=256, out_dim=embedding_dim)

    def forward_backbone(self, x):
        feat = self.backbone(x)
        emb = self.encoder(feat)
        return emb

    def forward_ssl(self, x1, x2):
        f1 = self.forward_backbone(x1)
        f2 = self.forward_backbone(x2)
        z1 = self.projector(f1)
        z2 = self.projector(f2)
        p1 = self.predictor(z1)
        p2 = self.predictor(z2)
        return p1, p2, z1.detach(), z2.detach()

class SSLFineTuneModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.neck(feat)
        logits = self.head(self.dropout(emb))
        return logits, emb

# =========================================================
# ViT CE model
# =========================================================
class ViTCEModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )
        backbone_out = self.backbone.num_features
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(backbone_out, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        logits = self.head(self.dropout(feat))
        return logits, feat

# =========================================================
# Common predict helpers
# =========================================================
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
supcon_criterion = SupConLoss(temperature=cfg.temperature)

@torch.no_grad()
def predict_probs_standard(model, loader, device):
    model.eval()
    all_probs, all_ids = [], []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            if MODEL_KIND == "arcface":
                ce_logits, arc_logits, _ = model(images, labels=None)
                logits = 0.5 * ce_logits + 0.5 * arc_logits
            else:
                logits, _ = model(images)

            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(ids.tolist() if torch.is_tensor(ids) else list(ids))

    return np.array(all_ids), np.concatenate(all_probs, axis=0)

@torch.no_grad()
def predict_with_tta_standard(model, df, device, is_test=True, tta_count=4):
    probs_list, labels_ref, ids_ref = [], None, None

    for tta_id in range(tta_count):
        ds = BaseImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False
        )

        if is_test:
            ids, probs = predict_probs_standard(model, dl, device)
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs, fold_labels = [], []
            for images, labels in dl:
                images = images.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    if MODEL_KIND == "arcface":
                        ce_logits, arc_logits, _ = model(images, labels=None)
                        logits = 0.5 * ce_logits + 0.5 * arc_logits
                    else:
                        logits, _ = model(images)
                    probs = torch.softmax(logits, dim=1)

                fold_probs.append(probs.cpu().numpy())
                fold_labels.append(labels.numpy())

            labels_ref = np.concatenate(fold_labels, axis=0)
            probs_list.append(np.concatenate(fold_probs, axis=0))

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)
    return (ids_ref, mean_probs) if is_test else (mean_probs, labels_ref)

# =========================================================
# ArcFace train/valid
# =========================================================
def train_arcface_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_ce = total_arc = total_correct = total_count = 0

    arc_criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (fused_logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_ce / total_count, total_arc / total_count, total_correct / total_count

@torch.no_grad()
def valid_arcface_one_epoch(model, loader, device):
    model.eval()
    arc_criterion = nn.CrossEntropyLoss()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(fused_logits, dim=1)
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# SupCon train/valid
# =========================================================
def train_supcon_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_ce = total_sup = total_correct = total_count = 0

    for image1, image2, labels in loader:
        image1 = image1.to(device, non_blocking=True)
        image2 = image2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits1, emb1 = model(image1)
            _, emb2 = model(image2)
            ce_loss = ce_criterion(logits1, labels)
            sup_loss = supcon_criterion(torch.stack([emb1, emb2], dim=1), labels)
            loss = cfg.ce_weight_sup * ce_loss + cfg.supcon_weight * sup_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_sup += sup_loss.item() * labels.size(0)
        total_correct += (logits1.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_ce / total_count, total_sup / total_count, total_correct / total_count

@torch.no_grad()
def valid_supcon_one_epoch(model, loader, device):
    model.eval()
    total_loss = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# ViT CE train/valid
# =========================================================
def train_vit_ce_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            loss = ce_criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def valid_vit_ce_one_epoch(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_labels, axis=0),
    )

# =========================================================
# SSL pretrain + fine-tune
# =========================================================
def negative_cosine_similarity(p, z):
    p = F.normalize(p, dim=1)
    z = F.normalize(z, dim=1)
    return -(p * z).sum(dim=1).mean()

def ssl_pretrain(model, loader, epochs, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    backbone_params = list(model.backbone.parameters()) + list(model.encoder.parameters())
    proj_params = list(model.projector.parameters()) + list(model.predictor.parameters())

    optimizer = torch.optim.AdamW(
        [{"params": backbone_params, "lr": cfg.ssl_lr_backbone},
         {"params": proj_params, "lr": cfg.ssl_lr_proj}],
        weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(loader), eta_min=cfg.min_lr
    )

    for epoch in range(epochs):
        running_loss, total = 0.0, 0
        t0 = time.time()

        for x1, x2 in loader:
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                p1, p2, z1, z2 = model.forward_ssl(x1, x2)
                loss = 0.5 * negative_cosine_similarity(p1, z2) + 0.5 * negative_cosine_similarity(p2, z1)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item() * x1.size(0)
            total += x1.size(0)

        print(f"[SSL] Epoch {epoch+1:02d}/{epochs} | ssl_loss={running_loss/total:.4f} | time={time.time()-t0:.1f}s")

    return model

def train_ssl_finetune_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_correct = total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            loss = ce_criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def valid_ssl_finetune_one_epoch(model, loader, device):
    model.eval()
    total_loss = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            loss = ce_criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )

# =========================================================
# Optional SSL pretrain
# =========================================================
ssl_backbone_state = None
ssl_encoder_state = None

if MODEL_KIND == "ssl":
    print("\n" + "=" * 90)
    print("SSL PRETRAINING ON TRAIN ONLY")
    print("=" * 90)

    ssl_ds = SSLDataset(train_df, transform=get_ssl_transforms(cfg.image_size))
    ssl_loader = DataLoader(
        ssl_ds,
        batch_size=cfg.ssl_bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True
    )

    ssl_model = SSLPretrainModel(cfg.model_name, embedding_dim=cfg.embedding_dim).to(cfg.device)
    ssl_model = ssl_pretrain(ssl_model, ssl_loader, epochs=cfg.pre_epochs, device=cfg.device)

    ssl_backbone_state = copy.deepcopy(ssl_model.backbone.state_dict())
    ssl_encoder_state = copy.deepcopy(ssl_model.encoder.state_dict())

    del ssl_ds, ssl_loader, ssl_model
    gc.collect()
    torch.cuda.empty_cache()

# =========================================================
# CV
# =========================================================
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    print("\n" + "=" * 90)
    print(f"FOLD {fold+1}/{cfg.n_splits}")
    print("=" * 90)

    trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
    val_df = train_df.iloc[va_idx].reset_index(drop=True)

    if MODEL_KIND == "arcface":
        train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = ArcFaceModel(
            cfg.model_name, cfg.num_classes,
            embedding_dim=cfg.embedding_dim,
            arc_s=cfg.arc_s, arc_m=cfg.arc_m,
            dropout=cfg.dropout
        ).to(cfg.device)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.ce_head.parameters()) + list(model.arc_head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()
            tr_loss, tr_ce, tr_arc, tr_acc = train_arcface_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_ce, va_arc, va_acc, _, _ = valid_arcface_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, arc={tr_arc:.4f}) | "
                f"train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} (ce={va_ce:.4f}, arc={va_arc:.4f}) | "
                f"valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "supcon":
        train_ds = SupConDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False, two_views=True)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = SupConModel(cfg.model_name, cfg.num_classes, embedding_dim=cfg.embedding_dim, dropout=cfg.dropout).to(cfg.device)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.ce_head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()
            tr_loss, tr_ce, tr_sup, tr_acc = train_supcon_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_ce, va_acc, _, _ = valid_supcon_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, supcon={tr_sup:.4f}) | "
                f"train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "ssl":
        train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
        valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

        train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
        valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

        model = SSLFineTuneModel(cfg.model_name, cfg.num_classes, embedding_dim=cfg.embedding_dim, dropout=cfg.dropout).to(cfg.device)

        model.backbone.load_state_dict(copy.deepcopy(ssl_backbone_state))
        try:
            model.neck[0].weight.data.copy_(ssl_encoder_state["0.weight"])
            model.neck[0].bias.data.copy_(ssl_encoder_state["0.bias"])
            model.neck[1].weight.data.copy_(ssl_encoder_state["1.weight"])
            model.neck[1].bias.data.copy_(ssl_encoder_state["1.bias"])
            model.neck[1].running_mean.data.copy_(ssl_encoder_state["1.running_mean"])
            model.neck[1].running_var.data.copy_(ssl_encoder_state["1.running_var"])
        except Exception as e:
            print("partial neck transfer skipped:", e)

        backbone_params = list(model.backbone.parameters())
        head_params = list(model.neck.parameters()) + list(model.head.parameters())

        optimizer = torch.optim.AdamW(
            [{"params": backbone_params, "lr": cfg.lr_backbone},
             {"params": head_params, "lr": cfg.lr_head}],
            weight_decay=cfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg.ft_epochs * len(train_loader), eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.ft_epochs):
            t0 = time.time()
            tr_loss, tr_acc = train_ssl_finetune_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
            va_loss, va_acc, _, _ = valid_ssl_finetune_one_epoch(model, valid_loader, cfg.device)

            print(
                f"Epoch {epoch+1:02d}/{cfg.ft_epochs} | "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    elif MODEL_KIND == "vit_ce":
        train_ds = BaseImageDataset(
            trn_df,
            transform=get_train_transforms(cfg.image_size),
            is_test=False
        )
        valid_ds = BaseImageDataset(
            val_df,
            transform=get_valid_transforms(cfg.image_size),
            is_test=False
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=cfg.train_bs,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=True
        )
        valid_loader = DataLoader(
            valid_ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True
        )

        model = ViTCEModel(
            model_name=cfg.vit_model_name,
            num_classes=cfg.num_classes,
            dropout=cfg.dropout
        ).to(cfg.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr_backbone,
            weight_decay=cfg.weight_decay
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.epochs * len(train_loader),
            eta_min=cfg.min_lr
        )

        stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
        best_acc = -1.0
        best_path = best_model_path(fold)

        for epoch in range(cfg.epochs):
            t0 = time.time()

            tr_loss, tr_acc = train_vit_ce_one_epoch(
                model, train_loader, optimizer, scheduler, cfg.device
            )
            va_loss, va_acc, _, _ = valid_vit_ce_one_epoch(
                model, valid_loader, cfg.device
            )

            print(
                f"Epoch {epoch+1:02d}/{cfg.epochs} | "
                f"train_loss={tr_loss:.4f} | train_acc={tr_acc:.4f} | "
                f"valid_loss={va_loss:.4f} | valid_acc={va_acc:.4f} | "
                f"time={time.time()-t0:.1f}s"
            )

            if va_acc > best_acc:
                best_acc = va_acc
                torch.save(model.state_dict(), best_path)
                print(f"  saved best -> {best_path}")

            if stopper.step(va_acc):
                print(f"  early stopping at epoch {epoch+1}")
                break

    else:
        raise ValueError("MODEL_KIND must be one of: arcface / supcon / ssl / vit_ce")

    model.load_state_dict(torch.load(best_path, map_location=cfg.device))

    val_tta_probs, val_tta_labels = predict_with_tta_standard(model, val_df, cfg.device, is_test=False, tta_count=cfg.tta_count)
    val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
    print(f"Fold {fold+1} TTA valid acc: {val_tta_acc:.5f}")
    oof_probs[va_idx] = val_tta_probs
    np.save(fold_val_probs_path(fold), val_tta_probs)

    _, fold_test_probs = predict_with_tta_standard(model, test_df, cfg.device, is_test=True, tta_count=cfg.tta_count)
    test_probs += fold_test_probs / cfg.n_splits
    np.save(fold_test_probs_path(fold), fold_test_probs)

    fold_scores.append({
        "fold": fold,
        "tta_valid_acc": float(val_tta_acc),
        "best_valid_acc": float(best_acc),
    })

    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    torch.cuda.empty_cache()

# =========================================================
# Save outputs
# =========================================================
oof_preds = oof_probs.argmax(1)
oof_acc = accuracy_score(train_df["label"].values, oof_preds)
print("\n" + "=" * 90)
print(f"OOF ACCURACY: {oof_acc:.6f}")
print("=" * 90)

np.save(final_oof_probs_path(), oof_probs)
np.save(final_test_probs_path(), test_probs)

submission = test_df[["ID"]].copy()
submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
submission.to_csv(final_submission_path(), index=False)

pd.DataFrame(fold_scores).to_csv(fold_scores_path(), index=False)

meta = {
    "model_kind": MODEL_KIND,
    "output_prefix": OUTPUT_PREFIX,
    "cfg": asdict(cfg),
    "oof_accuracy": float(oof_acc),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}
with open(metadata_path(), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"saved -> {final_oof_probs_path()}")
print(f"saved -> {final_test_probs_path()}")
print(f"saved -> {final_submission_path()}")
print(f"saved -> {fold_scores_path()}")
print(f"saved -> {metadata_path()}")
print(submission.head())

### 3. convnextv2_base_512_arcface_ce60arc40

In [ ]:
%%time
# =========================================================
# Clean training script for workshop submission
#   Single active configuration:
#     ConvNeXtV2 + CE head + ArcFace head
#     5-fold CV, TTA inference, OOF/test probability saving
#
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm


# =========================================================
# Config
# =========================================================
# OUTPUT_PREFIX is prepended to every output artifact.
# Keeping it unique prevents accidental overwriting when several
# experiments are executed in the same save directory.
OUTPUT_PREFIX = "convnextv2_base_512_arcface_ce60arc40"


@dataclass
# Central configuration block.
# All hyperparameters that affect training, inference, and output paths
# are collected here so that the experiment can be reproduced from metadata.
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"
    save_dir: str = "./ensemble_artifacts"

    # ConvNeXtV2 checkpoint used as the image encoder.
    # The model is pretrained and then fine-tuned for 10-way source attribution.
    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"
    image_size: int = 512
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20
    train_bs: int = 8
    valid_bs: int = 16
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    arc_s: float = 30.0
    arc_m: float = 0.30

    # Training loss: CE loss + ArcFace loss.
    # These weights reproduce the CE:ArcFace = 60:40 setting.
    ce_weight: float = 0.6
    arc_weight: float = 0.4

    # Inference logits: CE logits + margin-free ArcFace cosine logits
    infer_ce_weight: float = 0.5
    infer_arc_weight: float = 0.5

    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0
    tta_count: int = 4

    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)


# =========================================================
# Output helpers
# =========================================================
# The following helpers define all output filenames in one place.
# This makes the saved checkpoints, probabilities, submission file,
# and metadata consistent across folds.
def out_path(name: str) -> str:
    return os.path.join(cfg.save_dir, f"{OUTPUT_PREFIX}_{name}")


def best_model_path(fold: int) -> str:
    return out_path(f"best_fold{fold}.pth")


def fold_val_probs_path(fold: int) -> str:
    return out_path(f"fold{fold}_val_probs.npy")


def fold_test_probs_path(fold: int) -> str:
    return out_path(f"fold{fold}_test_probs.npy")


def final_oof_probs_path() -> str:
    return out_path("oof_probs.npy")


def final_test_probs_path() -> str:
    return out_path("test_probs.npy")


def final_submission_path() -> str:
    return out_path("submission.csv")


def metadata_path() -> str:
    return out_path("metadata.json")


def fold_scores_path() -> str:
    return out_path("fold_scores.csv")


# =========================================================
# Utilities
# =========================================================
# Seed Python, NumPy, and PyTorch RNGs.
# cuDNN benchmark is kept enabled below to preserve the original speed-oriented setup.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism is not forced because this notebook prioritizes speed.
    torch.backends.cudnn.benchmark = True


# Convert dataset-relative paths into paths that can be opened by OpenCV.
def make_abs_path(p: str) -> str:
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)


# AMP is enabled only on CUDA devices.
# This avoids CPU autocast behavior changing between environments.
def amp_enabled() -> bool:
    return bool(cfg.use_amp and cfg.device.startswith("cuda"))


# Normalize inference weights before fusing CE and ArcFace logits.
# This preserves the ratio even if the two values do not sum exactly to one.
def normalize_infer_weights(ce_w: float, arc_w: float) -> tuple[float, float]:
    total = ce_w + arc_w
    if total <= 0:
        raise ValueError("infer_ce_weight + infer_arc_weight must be > 0")
    return ce_w / total, arc_w / total


set_seed(cfg.seed)


# =========================================================
# Read data
# =========================================================
# The competition CSVs are expected to contain image paths and IDs.
# Training labels may be named either "y" or "TARGET".
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train_df must contain a label column named 'y' or 'TARGET'.")

if "path" not in train_df.columns or "path" not in test_df.columns:
    raise ValueError("Both train and test CSV files must contain a 'path' column.")

if "ID" not in test_df.columns:
    raise ValueError("test_df must contain an 'ID' column.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("OUTPUT_PREFIX:", OUTPUT_PREFIX)
print("device:", cfg.device)
print("AMP enabled:", amp_enabled())
print("train/test shape:", train_df.shape, test_df.shape)
print("class counts:")
print(train_df["label"].value_counts().sort_index())

print("\nActive configuration:")
print("model_name:", cfg.model_name)
print("image_size:", cfg.image_size)
print("CE:ArcFace loss weight:", cfg.ce_weight, cfg.arc_weight)
print("inference CE:ArcFace logit weight:", cfg.infer_ce_weight, cfg.infer_arc_weight)
print("ArcFace s, m:", cfg.arc_s, cfg.arc_m)


# =========================================================
# Transforms
# =========================================================
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


# Training-time augmentation.
# This version is intentionally conservative: it keeps the source artifacts
# mostly intact while adding mild geometric and photometric variation.
def get_train_transforms(img_size: int = 512):
    return A.Compose([
        # Mild random crop and resize. This preserves most facial structure
        # while exposing the model to small framing differences.
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.85, 1.0),
            ratio=(0.95, 1.05),
            interpolation=cv2.INTER_AREA,
            p=1.0,
        ),
        A.HorizontalFlip(p=0.5),
        # Small rotations and weak brightness/contrast changes approximate
        # possible post-processing without destroying generator-specific traces.
        A.Rotate(limit=5, border_mode=cv2.BORDER_REFLECT_101, p=0.25),
        A.RandomBrightnessContrast(
            brightness_limit=0.06,
            contrast_limit=0.06,
            p=0.20,
        ),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# Validation preprocessing is deterministic.
# No augmentation is applied here except resizing and normalization.
def get_valid_transforms(img_size: int = 512):
    return A.Compose([
        A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# Test-time augmentation views.
# The four views are averaged later to reduce sensitivity to small flips/crops.
def get_tta_transforms(img_size: int = 512, tta_id: int = 0):
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    if tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    if tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    return A.Compose([
        A.Resize(int(img_size * 1.05), int(img_size * 1.05), interpolation=cv2.INTER_AREA),
        A.CenterCrop(height=img_size, width=img_size, p=1.0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# =========================================================
# Dataset
# =========================================================
# Dataset wrapper shared by training, validation, and test inference.
# For test data it returns image IDs; for training/validation it returns labels.
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        # OpenCV loads images in BGR order, so we convert them to RGB
        # before applying Albumentations transforms.
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])

        return image, int(row["label"])


# =========================================================
# Early stopping
# =========================================================
# Simple early stopping utility.
# The best checkpoint is selected by validation accuracy, matching the experiment.
class EarlyStopping:
    def __init__(self, patience: int = 4, min_delta: float = 1e-4, mode: str = "max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False

        if self.mode == "max":
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True

        return self.should_stop


# =========================================================
# CE + ArcFace model
# =========================================================
# ArcFace classification head.
# During training, the target class logit receives an angular margin.
# During inference, labels are not available and margin-free cosine logits are used.
class ArcMarginProduct(nn.Module):
    """
    ArcFace head.

    labels is not None:
        returns ArcFace-margin logits for training loss.
    labels is None:
        returns margin-free cosine logits for inference.
    """
    def __init__(self, in_features: int, out_features: int, s: float = 30.0, m: float = 0.30):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor | None = None) -> torch.Tensor:
        # Cosine similarity between normalized embeddings and normalized class weights.
        cosine = F.linear(
            F.normalize(embeddings),
            F.normalize(self.weight),
        ).clamp(-1.0 + 1e-7, 1.0 - 1e-7)

        if labels is None:
            # Inference mode: no target labels are available, so no angular margin is applied.
            return cosine * self.s

        labels = labels.view(-1).long()

        # Compute cos(theta + m) for the target class.
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        return logits * self.s


# Full network: ConvNeXtV2 backbone, embedding neck, CE head, and ArcFace head.
# The CE and ArcFace heads share the same embedding.
class ArcFaceCEModel(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_classes: int,
        embedding_dim: int = 512,
        arc_s: float = 30.0,
        arc_m: float = 0.30,
        dropout: float = 0.2,
    ):
        super().__init__()

        # num_classes=0 removes the default classifier and returns pooled features.
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )

        backbone_out = self.backbone.num_features

        # The neck maps backbone features to a compact embedding used by both heads.
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(),
        )

        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(
            embedding_dim,
            num_classes,
            s=arc_s,
            m=arc_m,
        )

    def forward(self, x: torch.Tensor, labels: torch.Tensor | None = None):
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb_drop = self.dropout(emb)

        ce_logits = self.ce_head(emb_drop)
        arc_logits = self.arc_head(emb_drop, labels)

        return ce_logits, arc_logits, emb


# Fuse the CE head and ArcFace head logits for monitoring and inference.
def fuse_logits(ce_logits: torch.Tensor, arc_logits: torch.Tensor) -> torch.Tensor:
    ce_w, arc_w = normalize_infer_weights(cfg.infer_ce_weight, cfg.infer_arc_weight)
    return ce_w * ce_logits + arc_w * arc_logits


# =========================================================
# Train / validation / prediction
# =========================================================
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
arc_criterion = nn.CrossEntropyLoss()


# One supervised training epoch.
# The model is optimized with a weighted sum of standard CE loss and ArcFace loss.
def train_one_epoch(model, loader, optimizer, scheduler, device: str):
    model.train()

    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled())

    total_loss = 0.0
    total_ce = 0.0
    total_arc = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=amp_enabled()):
            # labels are passed to the ArcFace head so that the angular margin
            # is applied only to the correct class during training.
            ce_logits, arc_logits, _ = model(images, labels=labels)

            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)

            loss = cfg.ce_weight * ce_loss + cfg.arc_weight * arc_loss

            # Training accuracy is only a rough monitor because arc_logits
            # include the ArcFace training margin.
            fused_logits = fuse_logits(ce_logits, arc_logits)

        # Standard mixed-precision training step with gradient clipping.
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_ce += ce_loss.item() * bs
        total_arc += arc_loss.item() * bs
        total_correct += (fused_logits.argmax(1) == labels).sum().item()
        total_count += bs

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
    )


@torch.no_grad()
# Validation after each epoch.
# To preserve reproducibility with the submitted run, this function keeps
# the original label-conditioned ArcFace validation behavior.
def valid_one_epoch(model, loader, device: str):
    model.eval()

    total_loss = 0.0
    total_ce = 0.0
    total_arc = 0.0
    total_correct = 0
    total_count = 0

    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=amp_enabled()):
            # Keep the original validation behavior for reproducibility:
            # both validation loss and validation accuracy use label-conditioned
            # ArcFace logits, exactly as in the submitted experimental script.
            ce_logits, arc_logits, _ = model(images, labels=labels)

            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight * ce_loss + cfg.arc_weight * arc_loss

            fused_logits = fuse_logits(ce_logits, arc_logits)
            probs = torch.softmax(fused_logits, dim=1)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_ce += ce_loss.item() * bs
        total_arc += arc_loss.item() * bs
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += bs

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_labels, axis=0),
    )


@torch.no_grad()
# Predict class probabilities for one deterministic view.
# Unlike training/validation loss, inference uses labels=None, i.e.,
# margin-free ArcFace logits.
def predict_probs(model, loader, device: str):
    model.eval()

    all_probs = []
    all_targets = []

    for images, targets in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=amp_enabled()):
            # labels=None switches the ArcFace head to margin-free inference mode.
            ce_logits, arc_logits, _ = model(images, labels=None)
            fused_logits = fuse_logits(ce_logits, arc_logits)
            probs = torch.softmax(fused_logits, dim=1)

        all_probs.append(probs.cpu().numpy())

        if torch.is_tensor(targets):
            all_targets.extend(targets.cpu().numpy().tolist())
        else:
            all_targets.extend(list(targets))

    return np.asarray(all_targets), np.concatenate(all_probs, axis=0)


@torch.no_grad()
# Average predictions over multiple TTA views.
# For validation it returns probabilities and labels; for test it returns
# probabilities and image IDs.
def predict_with_tta(model, df: pd.DataFrame, device: str, is_test: bool, tta_count: int = 4):
    probs_list = []
    target_ref = None

    for tta_id in range(tta_count):
        ds = ImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test,
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )

        targets, probs = predict_probs(model, dl, device)
        probs_list.append(probs)

        if target_ref is None:
            target_ref = targets

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)

    if is_test:
        return target_ref, mean_probs

    return mean_probs, target_ref


# Construct a fresh model for each fold.
def build_model() -> ArcFaceCEModel:
    return ArcFaceCEModel(
        model_name=cfg.model_name,
        num_classes=cfg.num_classes,
        embedding_dim=cfg.embedding_dim,
        arc_s=cfg.arc_s,
        arc_m=cfg.arc_m,
        dropout=cfg.dropout,
    ).to(cfg.device)


# Use separate learning rates for the pretrained backbone and newly added heads.
# The cosine scheduler is stepped once per mini-batch.
def build_optimizer_and_scheduler(model, train_loader):
    backbone_params = list(model.backbone.parameters())
    head_params = (
        list(model.neck.parameters())
        + list(model.ce_head.parameters())
        + list(model.arc_head.parameters())
    )

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": cfg.lr_backbone},
            {"params": head_params, "lr": cfg.lr_head},
        ],
        weight_decay=cfg.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.epochs * len(train_loader),
        eta_min=cfg.min_lr,
    )

    return optimizer, scheduler


# =========================================================
# Cross-validation
# =========================================================
# Five stratified folds are used so that each fold keeps the original class balance.
# Fold-wise validation probabilities are written into oof_probs.
skf = StratifiedKFold(
    n_splits=cfg.n_splits,
    shuffle=True,
    random_state=cfg.seed,
)

oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    # Split the dataframe first; each fold builds its own datasets,
    # dataloaders, model, optimizer, and scheduler.
    print("\n" + "=" * 90)
    print(f"FOLD {fold + 1}/{cfg.n_splits}")
    print("=" * 90)

    trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
    val_df = train_df.iloc[va_idx].reset_index(drop=True)

    train_ds = ImageDataset(
        trn_df,
        transform=get_train_transforms(cfg.image_size),
        is_test=False,
    )
    valid_ds = ImageDataset(
        val_df,
        transform=get_valid_transforms(cfg.image_size),
        is_test=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=False,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg.valid_bs,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=False,
    )

    model = build_model()
    optimizer, scheduler = build_optimizer_and_scheduler(model, train_loader)

    stopper = EarlyStopping(
        patience=cfg.patience,
        min_delta=cfg.min_delta,
        mode="max",
    )

    best_acc = -1.0
    best_path = best_model_path(fold)

    for epoch in range(cfg.epochs):
        t0 = time.time()

        # Train one epoch, then evaluate on the validation fold.
        tr_loss, tr_ce, tr_arc, tr_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            cfg.device,
        )

        va_loss, va_ce, va_arc, va_acc, _, _ = valid_one_epoch(
            model,
            valid_loader,
            cfg.device,
        )

        print(
            f"Epoch {epoch + 1:02d}/{cfg.epochs} | "
            f"train_loss={tr_loss:.4f} "
            f"(ce={tr_ce:.4f}, arc={tr_arc:.4f}) | "
            f"train_acc={tr_acc:.4f} | "
            f"valid_loss={va_loss:.4f} "
            f"(ce={va_ce:.4f}, arc={va_arc:.4f}) | "
            f"valid_acc={va_acc:.4f} | "
            f"time={time.time() - t0:.1f}s"
        )

        # Save the checkpoint selected by the validation accuracy used in this script.
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), best_path)
            print(f"  saved best -> {best_path}")

        if stopper.step(va_acc):
            print(f"  early stopping at epoch {epoch + 1}")
            break

    # -----------------------------------------------------
    # Load best fold model and run TTA prediction
    # -----------------------------------------------------
    # Reload the best checkpoint before TTA validation and test prediction.
    model.load_state_dict(torch.load(best_path, map_location=cfg.device))
    model.eval()

    # Store OOF predictions from TTA validation for later ensemble use.
    val_tta_probs, val_tta_labels = predict_with_tta(
        model,
        val_df,
        cfg.device,
        is_test=False,
        tta_count=cfg.tta_count,
    )

    # Average test probabilities across folds to form the final prediction.
    _, fold_test_probs = predict_with_tta(
        model,
        test_df,
        cfg.device,
        is_test=True,
        tta_count=cfg.tta_count,
    )

    val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
    print(f"Fold {fold + 1} TTA valid acc: {val_tta_acc:.5f}")

    oof_probs[va_idx] = val_tta_probs.astype(np.float32)
    test_probs += fold_test_probs.astype(np.float32) / cfg.n_splits

    np.save(fold_val_probs_path(fold), val_tta_probs.astype(np.float32))
    np.save(fold_test_probs_path(fold), fold_test_probs.astype(np.float32))

    fold_scores.append({
        "fold": int(fold),
        "best_valid_acc": float(best_acc),
        "tta_valid_acc": float(val_tta_acc),
    })

    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# Save final outputs
# =========================================================
# The saved probability arrays can be reused for ensembling, while
# submission.csv is the direct competition submission file.
oof_preds = oof_probs.argmax(axis=1)
oof_acc = accuracy_score(train_df["label"].values, oof_preds)

print("\n" + "=" * 90)
print(f"OOF ACCURACY: {oof_acc:.6f}")
print("=" * 90)

np.save(final_oof_probs_path(), oof_probs)
np.save(final_test_probs_path(), test_probs.astype(np.float32))

submission = test_df[["ID"]].copy()
submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
submission.to_csv(final_submission_path(), index=False)

pd.DataFrame(fold_scores).to_csv(fold_scores_path(), index=False)

meta = {
    "description": "Single-mode ConvNeXtV2 CE + ArcFace training script.",
    "output_prefix": OUTPUT_PREFIX,
    "cfg": asdict(cfg),
    "oof_accuracy": float(oof_acc),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}

with open(metadata_path(), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"saved -> {final_oof_probs_path()}")
print(f"saved -> {final_test_probs_path()}")
print(f"saved -> {final_submission_path()}")
print(f"saved -> {fold_scores_path()}")
print(f"saved -> {metadata_path()}")
print(submission.head())


### 4. 512_arcface_base_postproc_aug_v1_inferacc

In [ ]:
%%time
# =========================================================
# Workshop submission code: ConvNeXtV2 + CE + ArcFace
# =========================================================
#   - post-processing-aware augmentation
#   - train accuracy computed from CE logits
#   - best checkpoint selected by valid_arcface_infer_acc
#   - TTA inference using labels=None and 0.5*CE + 0.5*ArcFace logits
#
# Execution flow:
#   1. Read train/test CSV files and build absolute image paths.
#   2. Train a ConvNeXtV2-based CE + ArcFace model with 5-fold CV.
#   3. Select the best checkpoint using inference-mode validation accuracy.
#   4. Run TTA inference for validation/test images.
#   5. Save OOF probabilities, test probabilities, fold scores, metadata,
#      and a submission CSV.
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm


# =========================================================
# Experiment identifier
# =========================================================
MODEL_KIND = "arcface"
OUTPUT_PREFIX = "512_arcface_base_postproc_aug_v1_inferacc"


# =========================================================
# Config
# =========================================================
# The configuration is centralized here so that the submitted experiment can
# be reproduced from a single place. Values that affect training, validation,
# inference, and output naming should be changed only with care.
@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"
    save_dir: str = "./ensemble_artifacts"

    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"
    image_size: int = 512
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20

    train_bs: int = 8
    valid_bs: int = 16
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    arc_s: float = 30.0
    arc_m: float = 0.30
    ce_weight_arc: float = 0.6
    arc_weight: float = 0.4

    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0

    tta_count: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)


# =========================================================
# Output helpers
# =========================================================
# All output filenames are derived from OUTPUT_PREFIX. This avoids accidental
# overwriting when multiple experiments are run in the same output directory.
def out_path(name):
    return os.path.join(cfg.save_dir, f"{OUTPUT_PREFIX}_{name}")


def best_model_path(fold):
    return out_path(f"best_fold{fold}.pth")


def fold_val_probs_path(fold):
    return out_path(f"fold{fold}_val_probs.npy")


def fold_test_probs_path(fold):
    return out_path(f"fold{fold}_test_probs.npy")


def final_oof_probs_path():
    return out_path("oof_probs.npy")


def final_test_probs_path():
    return out_path("test_probs.npy")


def final_submission_path():
    return out_path("submission.csv")


def metadata_path():
    return out_path("metadata.json")


def fold_scores_path():
    return out_path("fold_scores.csv")


# =========================================================
# Utils
# =========================================================
# Fix random seeds for Python, NumPy, and PyTorch. This improves reproducibility,
# although full determinism is still not guaranteed on every GPU/CUDA setting.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)


# Convert relative image paths from the CSV files into paths under data_root.
# Absolute paths are returned unchanged so the script also works with prebuilt paths.
def make_abs_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)


# =========================================================
# Read data
# =========================================================
# The competition data uses either `y` or `TARGET` as the class label column.
# The script normalizes this to a single `label` column used throughout training.
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train.csv must contain 'y' or 'TARGET'.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("MODEL_KIND:", MODEL_KIND)
print("OUTPUT_PREFIX:", OUTPUT_PREFIX)
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())


# =========================================================
# Transforms
# =========================================================
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


# Training augmentation is intentionally post-processing-aware. The parameters
# are preserved from the original run because changing them would change the score.
def get_train_transforms(img_size=384):
    return A.Compose([
        # Random crop and resizing.
        # The scale and ratio are kept exactly as in the original submitted code.
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.72, 1.0),
            ratio=(0.92, 1.08),
            interpolation=cv2.INTER_AREA,
            p=1.0,
        ),
        A.HorizontalFlip(p=0.5),

        # Small rotation. crop_border=True is intentionally preserved.
        A.Rotate(
            limit=6,
            interpolation=cv2.INTER_AREA,
            border_mode=cv2.BORDER_REFLECT_101,
            crop_border=True,
            p=0.25,
        ),

        # Brightness/contrast robustness.
        A.RandomBrightnessContrast(
            brightness_limit=0.10,
            contrast_limit=0.10,
            p=0.30,
        ),

        # Compression robustness.
        A.ImageCompression(
            quality_range=(55, 100),
            p=0.35,
        ),

        # Weak Gaussian blur.
        A.GaussianBlur(
            blur_limit=(3, 3),
            p=0.12,
        ),

        # Grayscale conversion.
        A.ToGray(p=0.12),

        # Resize / super-resolution artifact approximation.
        A.OneOf([
            A.Downscale(
                scale_range=(0.75, 0.95),
                interpolation_pair={
                    "downscale": cv2.INTER_AREA,
                    "upscale": cv2.INTER_CUBIC,
                },
                p=1.0,
            ),
            A.Resize(
                int(img_size * 0.85),
                int(img_size * 0.85),
                interpolation=cv2.INTER_AREA,
                p=1.0,
            ),
        ], p=0.12),

        A.Resize(img_size, img_size, interpolation=cv2.INTER_AREA),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# Validation preprocessing uses deterministic resizing and normalization only.
# No stochastic augmentation is applied in this path.
def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


# Test-time augmentation variants. These four deterministic views are averaged
# during validation/test inference. The IDs and transforms are part of the
# submitted behavior and are therefore kept unchanged.
def get_tta_transforms(img_size=384, tta_id=0):
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
            A.CenterCrop(height=img_size, width=img_size, p=1.0),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])


# =========================================================
# Dataset
# =========================================================
# Dataset wrapper for both training/validation and test images. For training and
# validation it returns (image, label); for test it returns (image, ID).
class BaseImageDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])
        return image, int(row["label"])


# =========================================================
# Common blocks
# =========================================================
# Simple early stopping helper. It tracks the best validation score and stops
# training when the score does not improve for `patience` epochs.
class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = score > self.best_score + self.min_delta if self.mode == "max" else score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


# =========================================================
# ArcFace model
# =========================================================
# ArcFace margin head. During training, labels are provided and the angular
# margin is applied only to the target class. During inference, labels=None is
# used and the head returns margin-free cosine logits scaled by `s`.
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        # Normalize embeddings and class weights before computing cosine logits.
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0, 1.0)
        if labels is None:
            # Inference mode: do not apply the ArcFace target-class margin.
            return cosine * self.s
        # Training mode: convert cos(theta) into cos(theta + margin) for the
        # target class, while keeping non-target class logits unchanged.
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        # Build a one-hot target mask so that the margin is applied only to the
        # ground-truth class for each sample.
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s


# Full image model. The ConvNeXtV2 backbone extracts image features, the neck
# maps them to a compact embedding, and two heads produce CE logits and ArcFace logits.
class ArcFaceModel(nn.Module):
    def __init__(self, model_name, num_classes, embedding_dim=512, arc_s=30.0, arc_m=0.30, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(),
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(embedding_dim, num_classes, s=arc_s, m=arc_m)

    def forward(self, x, labels=None):
        # Forward returns both heads plus the embedding so the same model can be
        # used for training, validation, and TTA inference.
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb = self.dropout(emb)
        ce_logits = self.ce_head(emb)
        if labels is not None:
            arc_logits = self.arc_head(emb, labels)
            return ce_logits, arc_logits, emb
        arc_logits = self.arc_head(emb, None)
        return ce_logits, arc_logits, emb


# =========================================================
# Losses and prediction helpers
# =========================================================
# Cross-entropy with label smoothing is used for the CE branch. The ArcFace
# branch uses a separate standard cross-entropy criterion inside the train/valid functions.
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)


@torch.no_grad()
# Predict class probabilities for a DataLoader. This function is used for test
# batches where IDs are returned. Inference always uses labels=None and averages
# CE logits and ArcFace cosine logits with fixed 0.5 / 0.5 weights.
def predict_probs_standard(model, loader, device):
    model.eval()
    all_probs, all_ids = [], []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels=None)
            logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(ids.tolist() if torch.is_tensor(ids) else list(ids))

    return np.array(all_ids), np.concatenate(all_probs, axis=0)


@torch.no_grad()
# Run deterministic TTA views and average probabilities. This is used for both
# validation OOF prediction and final test prediction.
def predict_with_tta_standard(model, df, device, is_test=True, tta_count=4):
    probs_list, labels_ref, ids_ref = [], None, None

    for tta_id in range(tta_count):
        ds = BaseImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test,
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )

        if is_test:
            ids, probs = predict_probs_standard(model, dl, device)
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs, fold_labels = [], []
            for images, labels in dl:
                images = images.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    ce_logits, arc_logits, _ = model(images, labels=None)
                    logits = 0.5 * ce_logits + 0.5 * arc_logits
                    probs = torch.softmax(logits, dim=1)

                fold_probs.append(probs.cpu().numpy())
                fold_labels.append(labels.numpy())

            labels_ref = np.concatenate(fold_labels, axis=0)
            probs_list.append(np.concatenate(fold_probs, axis=0))

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)
    return (ids_ref, mean_probs) if is_test else (mean_probs, labels_ref)


# =========================================================
# Train / validation
# =========================================================
# One training epoch. The loss is the weighted sum of CE loss and ArcFace loss.
# Note that training accuracy is intentionally computed from CE logits only,
# matching the original experiment.
def train_arcface_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)
    total_loss = total_ce = total_arc = total_correct = total_count = 0

    arc_criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            # The ArcFace branch receives labels here so the target-class margin
            # is applied for the ArcFace training loss.
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)

        # Intentionally preserved from the original code:
        # training accuracy is counted from CE logits, not from fused logits.
        total_correct += (ce_logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_ce / total_count, total_arc / total_count, total_correct / total_count


@torch.no_grad()
# Validation pass used to report validation loss and train-mode validation accuracy.
# This path supplies labels to the ArcFace head, matching the original code.
def valid_arcface_one_epoch(model, loader, device):
    model.eval()
    arc_criterion = nn.CrossEntropyLoss()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(fused_logits, dim=1)
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels),
    )


@torch.no_grad()
# Validation accuracy used for best-checkpoint selection. This is deliberately
# different from valid_arcface_one_epoch: labels=None is used so that validation
# matches the inference-time behavior.
def valid_arcface_infer_acc(model, loader, device):
    model.eval()

    total_correct = 0
    total_count = 0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels=None)
            logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(logits, dim=1)

        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_correct / total_count,
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_labels, axis=0),
    )


# =========================================================
# Cross validation
# =========================================================
# Five-fold stratified cross validation. Each fold trains a new model, saves its
# best checkpoint, and contributes one-fifth of the final test probability average.
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    print("\n" + "=" * 90)
    print(f"FOLD {fold+1}/{cfg.n_splits}")
    print("=" * 90)

    trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
    val_df = train_df.iloc[va_idx].reset_index(drop=True)

    # Build fold-specific datasets and dataloaders. Training uses stochastic
    # augmentation; validation uses deterministic preprocessing.
    train_ds = BaseImageDataset(
        trn_df,
        transform=get_train_transforms(cfg.image_size),
        is_test=False,
    )
    valid_ds = BaseImageDataset(
        val_df,
        transform=get_valid_transforms(cfg.image_size),
        is_test=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_bs,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg.valid_bs,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    # Initialize a fresh model for this fold. The backbone is pretrained by timm,
    # while the neck and classifier heads are trained for this task.
    model = ArcFaceModel(
        cfg.model_name,
        cfg.num_classes,
        embedding_dim=cfg.embedding_dim,
        arc_s=cfg.arc_s,
        arc_m=cfg.arc_m,
        dropout=cfg.dropout,
    ).to(cfg.device)

    # Use separate learning rates for the pretrained backbone and the newly added
    # neck/classification heads. This is kept identical to the original setting.
    backbone_params = list(model.backbone.parameters())
    head_params = list(model.neck.parameters()) + list(model.ce_head.parameters()) + list(model.arc_head.parameters())

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": cfg.lr_backbone},
            {"params": head_params, "lr": cfg.lr_head},
        ],
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.epochs * len(train_loader),
        eta_min=cfg.min_lr,
    )

    # Select the best epoch using inference-mode validation accuracy, not the
    # label-conditioned ArcFace validation accuracy.
    stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
    best_acc = -1.0
    best_path = best_model_path(fold)

    for epoch in range(cfg.epochs):
        t0 = time.time()
        tr_loss, tr_ce, tr_arc, tr_acc = train_arcface_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            cfg.device,
        )
        va_loss, va_ce, va_arc, va_acc_trainmode, _, _ = valid_arcface_one_epoch(
            model,
            valid_loader,
            cfg.device,
        )
        va_acc, _, _ = valid_arcface_infer_acc(
            model,
            valid_loader,
            cfg.device,
        )

        print(
            f"Epoch {epoch+1:02d}/{cfg.epochs} | "
            f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, arc={tr_arc:.4f}) | "
            f"train_acc={tr_acc:.4f} | "
            f"valid_loss={va_loss:.4f} (ce={va_ce:.4f}, arc={va_arc:.4f}) | "
            f"valid_acc={va_acc:.4f} | "
            f"trainmode_valid_acc={va_acc_trainmode:.4f} | "
            f"time={time.time()-t0:.1f}s"
        )

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), best_path)
            print(f"  saved best -> {best_path}")

        if stopper.step(va_acc):
            print(f"  early stopping at epoch {epoch+1}")
            break

    # Reload the best checkpoint before generating fold-level OOF and test probabilities.
    model.load_state_dict(torch.load(best_path, map_location=cfg.device))

    # Generate OOF probabilities for the validation fold using the same TTA
    # inference path as the final test prediction.
    val_tta_probs, val_tta_labels = predict_with_tta_standard(
        model,
        val_df,
        cfg.device,
        is_test=False,
        tta_count=cfg.tta_count,
    )
    val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
    print(f"Fold {fold+1} TTA valid acc: {val_tta_acc:.5f}")

    oof_probs[va_idx] = val_tta_probs
    np.save(fold_val_probs_path(fold), val_tta_probs)

    # Add this fold's test probabilities to the running average over folds.
    _, fold_test_probs = predict_with_tta_standard(
        model,
        test_df,
        cfg.device,
        is_test=True,
        tta_count=cfg.tta_count,
    )
    test_probs += fold_test_probs / cfg.n_splits
    np.save(fold_test_probs_path(fold), fold_test_probs)

    fold_scores.append({
        "fold": fold,
        "tta_valid_acc": float(val_tta_acc),
        "best_valid_acc": float(best_acc),
    })

    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    torch.cuda.empty_cache()


# =========================================================
# Save outputs
# =========================================================
# After all folds finish, compute OOF accuracy and write all artifacts needed for
# reproduction, ensembling, and competition submission.
oof_preds = oof_probs.argmax(1)
oof_acc = accuracy_score(train_df["label"].values, oof_preds)
print("\n" + "=" * 90)
print(f"OOF ACCURACY: {oof_acc:.6f}")
print("=" * 90)

np.save(final_oof_probs_path(), oof_probs)
np.save(final_test_probs_path(), test_probs)

submission = test_df[["ID"]].copy()
submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
submission.to_csv(final_submission_path(), index=False)

pd.DataFrame(fold_scores).to_csv(fold_scores_path(), index=False)

meta = {
    "model_kind": MODEL_KIND,
    "output_prefix": OUTPUT_PREFIX,
    "cfg": asdict(cfg),
    "oof_accuracy": float(oof_acc),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}
with open(metadata_path(), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"saved -> {final_oof_probs_path()}")
print(f"saved -> {final_test_probs_path()}")
print(f"saved -> {final_submission_path()}")
print(f"saved -> {fold_scores_path()}")
print(f"saved -> {metadata_path()}")
print(submission.head())


### 5. arcface_base_old_ft768_v1

In [ ]:
%%time
# =========================================================
# Workshop submission code: 768px fine-tuning from ArcFace checkpoint
# =========================================================
# Configuration:
#   - ConvNeXtV2 backbone
#   - CE head + ArcFace head
#   - 768x768 input fine-tuning
#   - initialization from fold-wise checkpoints produced by INIT_PREFIX
#   - gradient accumulation to keep the effective batch size larger than
#     the physical GPU batch size
#   - 5-fold CV, TTA inference, OOF/test probability saving
# =========================================================

# !pip -q install timm albumentations opencv-python-headless

import os
import gc
import cv2
import math
import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

# =========================================================
# Experiment identifiers
# =========================================================
# MODEL_KIND is kept as a fixed metadata field for compatibility with the
# original experiment logs.  The cleaned code no longer branches on it.
MODEL_KIND = "arcface"

# OUTPUT_PREFIX controls every generated file name.  Keeping it unchanged
# prevents accidental overwriting and preserves compatibility with the
# ensemble code that expects this prefix.
OUTPUT_PREFIX = "arcface_base_old_ft768_v1"

# INIT_PREFIX points to the previously trained ArcFace checkpoints used to
# initialize each fold before 768px fine-tuning.  For fold k, the code loads:
#   ./ensemble_artifacts/{INIT_PREFIX}_best_fold{k}.pth
INIT_PREFIX = "arcface_base"

# =========================================================
# Config
# =========================================================
@dataclass
class CFG:
    # General reproducibility and data settings.
    seed: int = 42
    num_classes: int = 10

    data_root: str = "./"
    train_csv: str = "./Data/training.csv"
    test_csv: str = "./Data/test.csv"

    save_dir: str = "./ensemble_artifacts"

    # ConvNeXtV2 backbone used for both training and fine-tuning.
    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"

    # 768px fine-tuning setting.  The model is convolutional with global
    # pooling, so the input resolution can be increased at fine-tuning time.
    image_size: int = 768
    embedding_dim: int = 512

    # Cross-validation and optimization length.
    n_splits: int = 5
    epochs: int = 20

    # Small physical batch sizes are used because 768px images require much
    # more memory.  Gradient accumulation restores a larger effective batch.
    train_bs: int = 2
    valid_bs: int = 4
    grad_accum_steps: int = 4

    num_workers: int = 0

    # Fine-tuning uses smaller learning rates than the original training run.
    lr_backbone: float = 2e-5
    lr_head: float = 5e-5

    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    # ArcFace and CE/ArcFace loss mixture.  These values are preserved from
    # the original experiment.
    arc_s: float = 30.0
    arc_m: float = 0.30
    ce_weight_arc: float = 0.6
    arc_weight: float = 0.4

    # Early stopping uses the normal validation accuracy from the ArcFace
    # validation function, matching the original 768px fine-tuning code.
    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0

    # Number of test-time augmentations used for OOF and test probability
    # generation after each fold checkpoint is selected.
    tta_count: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)

# =========================================================
# Output path helpers
# =========================================================
# All outputs are placed under cfg.save_dir and prefixed by OUTPUT_PREFIX.
# This keeps the fold checkpoints, fold probabilities, final OOF/test
# probabilities, submission, metadata, and fold score table grouped together.
def out_path(name):
    return os.path.join(cfg.save_dir, f"{OUTPUT_PREFIX}_{name}")


def best_model_path(fold):
    return out_path(f"best_fold{fold}.pth")


def init_model_path(fold):
    return os.path.join(cfg.save_dir, f"{INIT_PREFIX}_best_fold{fold}.pth")


def fold_val_probs_path(fold):
    return out_path(f"fold{fold}_val_probs.npy")


def fold_test_probs_path(fold):
    return out_path(f"fold{fold}_test_probs.npy")


def final_oof_probs_path():
    return out_path("oof_probs.npy")


def final_test_probs_path():
    return out_path("test_probs.npy")


def final_submission_path():
    return out_path("submission.csv")


def metadata_path():
    return out_path("metadata.json")


def fold_scores_path():
    return out_path("fold_scores.csv")


# =========================================================
# Utilities
# =========================================================
def set_seed(seed=42):
    # Set random seeds for Python, NumPy, and PyTorch.  This follows the
    # original code and improves reproducibility, although GPU kernels may
    # still introduce small nondeterminism depending on the environment.
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)


def make_abs_path(p):
    # Convert a CSV path entry into an absolute or data-root-relative path.
    if os.path.isabs(p):
        return p
    return os.path.join(cfg.data_root, p)


# =========================================================
# Read data
# =========================================================
train_df = pd.read_csv(cfg.train_csv)
test_df = pd.read_csv(cfg.test_csv)

# The competition data may use either 'y' or 'TARGET' as the label column.
# The cleaned script normalizes both cases to train_df['label'].
if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("train.csv must contain 'y' or 'TARGET'.")

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("MODEL_KIND:", MODEL_KIND)
print("OUTPUT_PREFIX:", OUTPUT_PREFIX)
print("INIT_PREFIX:", INIT_PREFIX)
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts().sort_index())

# =========================================================
# Transforms
# =========================================================
# ImageNet normalization is used because the ConvNeXtV2 backbone is pretrained
# on ImageNet-style RGB images.
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


def get_train_transforms(img_size=384):
    # Training augmentations are kept identical to the original 768px
    # fine-tuning code.  They expose the model to mild geometric, blur,
    # compression, grayscale, and resizing artifacts while preserving the
    # source-specific visual traces needed for attribution.
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.75, 1.0), ratio=(0.90, 1.10), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
        A.Sharpen(alpha=(0.1, 0.25), lightness=(0.9, 1.1), p=0.15),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.ToGray(p=0.07),
        A.OneOf([
            A.Downscale(scale_range=(0.5, 0.85), p=1.0),
            A.Resize(int(img_size * 0.85), int(img_size * 0.85), p=1.0),
        ], p=0.12),
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def get_valid_transforms(img_size=384):
    # Validation uses only resizing and normalization.  No stochastic
    # augmentation is applied during the epoch-level validation loop.
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def get_tta_transforms(img_size=384, tta_id=0):
    # Four deterministic TTA views are used after loading the best checkpoint:
    #   0: resize only
    #   1: horizontal flip
    #   2: center crop after resize
    #   3: slightly larger resize followed by center crop
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    elif tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
            A.CenterCrop(height=img_size, width=img_size, p=1.0),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])


# =========================================================
# Dataset
# =========================================================
class BaseImageDataset(Dataset):
    # Minimal image dataset for both training/validation and test inference.
    # In training mode it returns (image, label).  In test mode it returns
    # (image, ID), where ID is used to build the submission file.
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])
        return image, int(row["label"])


# =========================================================
# Early stopping
# =========================================================
class EarlyStopping:
    # Stop training when the monitored validation score does not improve for
    # cfg.patience epochs.  The monitored score is validation accuracy.
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = score > self.best_score + self.min_delta if self.mode == "max" else score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


# =========================================================
# ArcFace model
# =========================================================
class ArcMarginProduct(nn.Module):
    # ArcFace classification head.  During training, the target-class cosine
    # logit is modified by an angular margin.  During inference, labels are
    # unavailable, so the head returns margin-free cosine logits scaled by s.
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0, 1.0)
        if labels is None:
            return cosine * self.s
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s


class ArcFaceModel(nn.Module):
    # ConvNeXtV2 encoder followed by a compact embedding neck and two heads:
    #   - ce_head: ordinary cross-entropy classifier
    #   - arc_head: ArcFace margin classifier
    # Both heads are trained jointly and later fused for inference.
    def __init__(self, model_name, num_classes, embedding_dim=512, arc_s=30.0, arc_m=0.30, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        backbone_out = self.backbone.num_features
        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU()
        )
        self.dropout = nn.Dropout(dropout)
        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(embedding_dim, num_classes, s=arc_s, m=arc_m)

    def forward(self, x, labels=None):
        feat = self.backbone(x)
        emb = self.neck(feat)
        emb = self.dropout(emb)
        ce_logits = self.ce_head(emb)
        if labels is not None:
            arc_logits = self.arc_head(emb, labels)
            return ce_logits, arc_logits, emb
        arc_logits = self.arc_head(emb, None)
        return ce_logits, arc_logits, emb


# =========================================================
# Loss and prediction helpers
# =========================================================
ce_criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)


@torch.no_grad()
def predict_probs_standard(model, loader, device):
    # Predict class probabilities for a DataLoader that returns test IDs.
    # The CE and ArcFace inference logits are averaged exactly as in the
    # original code.
    model.eval()
    all_probs, all_ids = [], []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels=None)
            logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(ids.tolist() if torch.is_tensor(ids) else list(ids))

    return np.array(all_ids), np.concatenate(all_probs, axis=0)


@torch.no_grad()
def predict_with_tta_standard(model, df, device, is_test=True, tta_count=4):
    # Run deterministic TTA and average the probabilities.  For validation
    # folds, labels are also returned so that TTA accuracy and OOF predictions
    # can be computed.
    probs_list, labels_ref, ids_ref = [], None, None

    for tta_id in range(tta_count):
        ds = BaseImageDataset(
            df,
            transform=get_tta_transforms(cfg.image_size, tta_id),
            is_test=is_test
        )
        dl = DataLoader(
            ds,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False
        )

        if is_test:
            ids, probs = predict_probs_standard(model, dl, device)
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs, fold_labels = [], []
            for images, labels in dl:
                images = images.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg.use_amp):
                    ce_logits, arc_logits, _ = model(images, labels=None)
                    logits = 0.5 * ce_logits + 0.5 * arc_logits
                    probs = torch.softmax(logits, dim=1)

                fold_probs.append(probs.cpu().numpy())
                fold_labels.append(labels.numpy())

            labels_ref = np.concatenate(fold_labels, axis=0)
            probs_list.append(np.concatenate(fold_probs, axis=0))

        del ds, dl
        gc.collect()

    mean_probs = np.mean(probs_list, axis=0)
    return (ids_ref, mean_probs) if is_test else (mean_probs, labels_ref)


# =========================================================
# ArcFace training and validation
# =========================================================
def train_arcface_one_epoch(model, loader, optimizer, scheduler, device):
    # One training epoch with mixed precision and gradient accumulation.
    # loss_raw is used for logging, while loss is divided by grad_accum_steps
    # before backpropagation to keep accumulated gradients properly scaled.
    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    arc_criterion = nn.CrossEntropyLoss()

    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss_raw = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss
            loss = loss_raw / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        total_loss += loss_raw.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (ce_logits.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
    )


@torch.no_grad()
def valid_arcface_one_epoch(model, loader, device):
    # Epoch-level validation.  This intentionally uses label-conditioned
    # ArcFace logits, matching the original script and preserving the same
    # best-checkpoint selection behavior.
    model.eval()
    arc_criterion = nn.CrossEntropyLoss()

    total_loss = total_ce = total_arc = total_correct = total_count = 0
    all_probs, all_labels = [], []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = 0.5 * ce_logits + 0.5 * arc_logits
            probs = torch.softmax(fused_logits, dim=1)
            ce_loss = ce_criterion(ce_logits, labels)
            arc_loss = arc_criterion(arc_logits, labels)
            loss = cfg.ce_weight_arc * ce_loss + cfg.arc_weight * arc_loss

        total_loss += loss.item() * labels.size(0)
        total_ce += ce_loss.item() * labels.size(0)
        total_arc += arc_loss.item() * labels.size(0)
        total_correct += (probs.argmax(1) == labels).sum().item()
        total_count += labels.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels)
    )


# =========================================================
# Cross-validation training
# =========================================================
skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
oof_probs = np.zeros((len(train_df), cfg.num_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), cfg.num_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"])):
    print("\n" + "=" * 90)
    print(f"FOLD {fold+1}/{cfg.n_splits}")
    print("=" * 90)

    trn_df = train_df.iloc[tr_idx].reset_index(drop=True)
    val_df = train_df.iloc[va_idx].reset_index(drop=True)

    train_ds = BaseImageDataset(trn_df, transform=get_train_transforms(cfg.image_size), is_test=False)
    valid_ds = BaseImageDataset(val_df, transform=get_valid_transforms(cfg.image_size), is_test=False)

    train_loader = DataLoader(train_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    model = ArcFaceModel(
        cfg.model_name, cfg.num_classes,
        embedding_dim=cfg.embedding_dim,
        arc_s=cfg.arc_s, arc_m=cfg.arc_m,
        dropout=cfg.dropout
    ).to(cfg.device)

    # ---------------------------------------------------------
    # Initialize from the previous ArcFace checkpoint
    # ---------------------------------------------------------
    # If the fold-wise checkpoint exists, load it strictly.  If it does not
    # exist, the original code continues from the ImageNet-pretrained backbone;
    # this fallback behavior is preserved.
    init_path = init_model_path(fold)

    if os.path.exists(init_path):
        print(f"loading init checkpoint -> {init_path}")
        state = torch.load(init_path, map_location=cfg.device)
        model.load_state_dict(state, strict=True)
    else:
        print(f"WARNING: init checkpoint not found -> {init_path}")
        print("training from ImageNet pretrained weights instead")

    backbone_params = list(model.backbone.parameters())
    head_params = list(model.neck.parameters()) + list(model.ce_head.parameters()) + list(model.arc_head.parameters())

    optimizer = torch.optim.AdamW(
        [{"params": backbone_params, "lr": cfg.lr_backbone},
         {"params": head_params, "lr": cfg.lr_head}],
        weight_decay=cfg.weight_decay
    )

    # The original fine-tuning code sets T_max using the number of mini-batches,
    # even though scheduler.step() is called only after accumulated optimizer
    # updates.  This detail is kept unchanged for score reproducibility.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs * len(train_loader), eta_min=cfg.min_lr
    )

    stopper = EarlyStopping(cfg.patience, cfg.min_delta, mode="max")
    best_acc = -1.0
    best_path = best_model_path(fold)

    for epoch in range(cfg.epochs):
        t0 = time.time()
        tr_loss, tr_ce, tr_arc, tr_acc = train_arcface_one_epoch(model, train_loader, optimizer, scheduler, cfg.device)
        va_loss, va_ce, va_arc, va_acc, _, _ = valid_arcface_one_epoch(model, valid_loader, cfg.device)

        print(
            f"Epoch {epoch+1:02d}/{cfg.epochs} | "
            f"train_loss={tr_loss:.4f} (ce={tr_ce:.4f}, arc={tr_arc:.4f}) | "
            f"train_acc={tr_acc:.4f} | "
            f"valid_loss={va_loss:.4f} (ce={va_ce:.4f}, arc={va_arc:.4f}) | "
            f"valid_acc={va_acc:.4f} | time={time.time()-t0:.1f}s"
        )

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), best_path)
            print(f"  saved best -> {best_path}")

        if stopper.step(va_acc):
            print(f"  early stopping at epoch {epoch+1}")
            break

    # Load the selected checkpoint and compute TTA probabilities for the fold.
    model.load_state_dict(torch.load(best_path, map_location=cfg.device))

    val_tta_probs, val_tta_labels = predict_with_tta_standard(
        model,
        val_df,
        cfg.device,
        is_test=False,
        tta_count=cfg.tta_count,
    )
    val_tta_acc = accuracy_score(val_tta_labels, val_tta_probs.argmax(1))
    print(f"Fold {fold+1} TTA valid acc: {val_tta_acc:.5f}")

    oof_probs[va_idx] = val_tta_probs
    np.save(fold_val_probs_path(fold), val_tta_probs)

    _, fold_test_probs = predict_with_tta_standard(
        model,
        test_df,
        cfg.device,
        is_test=True,
        tta_count=cfg.tta_count,
    )
    test_probs += fold_test_probs / cfg.n_splits
    np.save(fold_test_probs_path(fold), fold_test_probs)

    fold_scores.append({
        "fold": fold,
        "tta_valid_acc": float(val_tta_acc),
        "best_valid_acc": float(best_acc),
    })

    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    torch.cuda.empty_cache()

# =========================================================
# Save outputs
# =========================================================
oof_preds = oof_probs.argmax(1)
oof_acc = accuracy_score(train_df["label"].values, oof_preds)
print("\n" + "=" * 90)
print(f"OOF ACCURACY: {oof_acc:.6f}")
print("=" * 90)

np.save(final_oof_probs_path(), oof_probs)
np.save(final_test_probs_path(), test_probs)

submission = test_df[["ID"]].copy()
submission["TARGET"] = test_probs.argmax(axis=1).astype(int)
submission.to_csv(final_submission_path(), index=False)

pd.DataFrame(fold_scores).to_csv(fold_scores_path(), index=False)

meta = {
    "model_kind": MODEL_KIND,
    "output_prefix": OUTPUT_PREFIX,
    "init_prefix": INIT_PREFIX,
    "cfg": asdict(cfg),
    "oof_accuracy": float(oof_acc),
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
}
with open(metadata_path(), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"saved -> {final_oof_probs_path()}")
print(f"saved -> {final_test_probs_path()}")
print(f"saved -> {final_submission_path()}")
print(f"saved -> {fold_scores_path()}")
print(f"saved -> {metadata_path()}")
print(submission.head())


### 6. Weighted ensemble

In [ ]:
# =========================================================
# Official final ensemble generation
#
# This cell reproduces the three official final submissions:
#
# Final A:
#   0.600 arcface_base
#   0.200 ssl_pre2_base
#   0.200 arcface_base_old_ft768_v1
#
# Final B:
#   0.675 arcface_base
#   0.200 ssl_pre2_base
#   0.125 512_arcface_base_postproc_aug_v1_inferacc
#
# Final C:
#   0.675 arcface_base
#   0.200 ssl_pre2_base
#   0.125 convnextv2_base_512_arcface_ce60arc40
#
# Required files:
#   ./ensemble_artifacts/{prefix}_test_probs.npy
#   ./Data/test.csv
#
# Outputs:
#   ./ensemble_artifacts/official_final_A_test_probs.npy
#   ./ensemble_artifacts/official_final_A_submission.csv
#   ./ensemble_artifacts/official_final_B_test_probs.npy
#   ./ensemble_artifacts/official_final_B_submission.csv
#   ./ensemble_artifacts/official_final_C_test_probs.npy
#   ./ensemble_artifacts/official_final_C_submission.csv
#   ./ensemble_artifacts/official_final_summary.csv
# =========================================================

import os
import json
import numpy as np
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
SAVE_DIR = "./ensemble_artifacts"
TEST_CSV = "./Data/test.csv"

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Official final ensemble definitions
# -----------------------------
OFFICIAL_ENSEMBLES = [
    {
        "name": "official_final_A",
        "description": "0.600 ArcFace base + 0.200 SSL + 0.200 768 CE+ArcFace",
        "prefixes": [
            "arcface_base",
            "ssl_pre2_base",
            "arcface_base_old_ft768_v1",
        ],
        "weights": [0.600, 0.200, 0.200],
        "public_lb": 0.995333,
        "private_lb": 0.992000,
    },
    {
        "name": "official_final_B",
        "description": "0.675 ArcFace base + 0.200 SSL + 0.125 512 post-processing-aware CE+ArcFace",
        "prefixes": [
            "arcface_base",
            "ssl_pre2_base",
            "512_arcface_base_postproc_aug_v1_inferacc",
        ],
        "weights": [0.675, 0.200, 0.125],
        "public_lb": 0.995333,
        "private_lb": 0.992000,
    },
    {
        "name": "official_final_C",
        "description": "0.675 ArcFace base + 0.200 SSL + 0.125 512 CE+ArcFace",
        "prefixes": [
            "arcface_base",
            "ssl_pre2_base",
            "convnextv2_base_512_arcface_ce60arc40",
        ],
        "weights": [0.675, 0.200, 0.125],
        "public_lb": 0.995333,
        "private_lb": 0.992666,
    },
]

# -----------------------------
# Load test IDs
# -----------------------------
test_df = pd.read_csv(TEST_CSV)

if "ID" not in test_df.columns:
    raise ValueError("test.csv must contain an 'ID' column.")

print("test_df:", test_df.shape)
print("SAVE_DIR:", SAVE_DIR)

# -----------------------------
# Helper functions
# -----------------------------
def load_probs(prefix: str, save_dir: str = SAVE_DIR) -> np.ndarray:
    """Load test probability file for one model prefix."""
    prefix = prefix.strip()
    path = os.path.join(save_dir, f"{prefix}_test_probs.npy")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing probability file: {path}\n"
            f"Expected file name pattern: {{prefix}}_test_probs.npy"
        )

    probs = np.load(path)

    if probs.ndim != 2:
        raise ValueError(f"{path} must be 2D, but got shape {probs.shape}")

    return probs.astype(np.float32)


def make_weighted_ensemble(prefixes, weights):
    """Create a weighted average of probability arrays."""
    prefixes = [p.strip() for p in prefixes]
    weights = np.asarray(weights, dtype=np.float32)

    if len(prefixes) != len(weights):
        raise ValueError(
            f"len(prefixes)={len(prefixes)} and len(weights)={len(weights)} must match."
        )

    weights = weights / weights.sum()

    prob_list = []
    for prefix in prefixes:
        probs = load_probs(prefix)
        print(f"  loaded {prefix}: shape={probs.shape}, dtype={probs.dtype}")
        prob_list.append(probs)

    base_shape = prob_list[0].shape
    for prefix, probs in zip(prefixes, prob_list):
        if probs.shape != base_shape:
            raise ValueError(
                f"Shape mismatch: {prefix} has {probs.shape}, expected {base_shape}"
            )

    if base_shape[0] != len(test_df):
        raise ValueError(
            f"Number of test rows mismatch: probs has {base_shape[0]}, "
            f"but test_df has {len(test_df)}"
        )

    ensemble_probs = np.zeros(base_shape, dtype=np.float32)
    for w, probs in zip(weights, prob_list):
        ensemble_probs += w * probs

    # Optional safety check: probabilities should be approximately normalized
    row_sums = ensemble_probs.sum(axis=1)
    print(
        f"  ensemble row-sum: min={row_sums.min():.6f}, "
        f"max={row_sums.max():.6f}, mean={row_sums.mean():.6f}"
    )

    return ensemble_probs, weights


# -----------------------------
# Generate official final submissions
# -----------------------------
summary_rows = []

for ens in OFFICIAL_ENSEMBLES:
    print("\n" + "=" * 100)
    print(ens["name"])
    print(ens["description"])
    print("=" * 100)

    prefixes = ens["prefixes"]
    weights = ens["weights"]

    print("prefixes:", prefixes)
    print("weights :", weights)

    ensemble_probs, normalized_weights = make_weighted_ensemble(prefixes, weights)
    preds = ensemble_probs.argmax(axis=1).astype(int)

    submission = test_df[["ID"]].copy()
    submission["TARGET"] = preds

    probs_out = os.path.join(SAVE_DIR, f"{ens['name']}_test_probs.npy")
    sub_out = os.path.join(SAVE_DIR, f"{ens['name']}_submission.csv")
    meta_out = os.path.join(SAVE_DIR, f"{ens['name']}_metadata.json")

    np.save(probs_out, ensemble_probs)
    submission.to_csv(sub_out, index=False)

    metadata = {
        "name": ens["name"],
        "description": ens["description"],
        "prefixes": prefixes,
        "weights": [float(w) for w in normalized_weights],
        "public_lb": ens["public_lb"],
        "private_lb": ens["private_lb"],
        "probs_out": probs_out,
        "submission_out": sub_out,
    }

    with open(meta_out, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"saved probs      -> {probs_out}")
    print(f"saved submission -> {sub_out}")
    print(f"saved metadata   -> {meta_out}")
    print(submission.head())

    summary_rows.append({
        "name": ens["name"],
        "description": ens["description"],
        "prefixes": " + ".join(prefixes),
        "weights": " + ".join([f"{w:.3f}" for w in normalized_weights]),
        "public_lb": ens["public_lb"],
        "private_lb": ens["private_lb"],
        "submission_file": os.path.basename(sub_out),
        "probs_file": os.path.basename(probs_out),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(SAVE_DIR, "official_final_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 100)
print("Official final summary")
print("=" * 100)
print(summary_df)
print(f"saved summary -> {summary_path}")